# House Price Prediction using Linear Regression

**Task:** Predict house prices based on square footage, number of bedrooms, and number of bathrooms.

This notebook covers the full workflow:
1. Load / generate data
2. Explore the data
3. Split into train/test sets
4. Train a Linear Regression model
5. Evaluate performance
6. Make predictions on new houses


## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

np.random.seed(42)


## 2. Load the Dataset

**If you have your own CSV file** (e.g. from Kaggle's House Prices dataset), replace the code in the next cell with:
```python
df = pd.read_csv('your_file.csv')
df = df[['GrLivArea', 'BedroomAbvGr', 'FullBath', 'SalePrice']]
df.columns = ['square_footage', 'bedrooms', 'bathrooms', 'price']
```
(Column names will vary by dataset — check `df.columns` first.)

For now, we'll generate a **synthetic dataset** so the notebook runs end-to-end without any external file.

In [ ]:
# --- Synthetic dataset generator (replace with pd.read_csv(...) for real data) ---
n_samples = 500

square_footage = np.random.normal(1800, 600, n_samples).clip(400, 5000)
bedrooms = np.random.randint(1, 6, n_samples)
bathrooms = np.random.randint(1, 4, n_samples)

# price roughly follows: base + $/sqft + bedroom value + bathroom value + noise
price = (
    50000
    + square_footage * 120
    + bedrooms * 8000
    + bathrooms * 12000
    + np.random.normal(0, 20000, n_samples)
).clip(50000, None)

df = pd.DataFrame({
    'square_footage': square_footage.round(0),
    'bedrooms': bedrooms,
    'bathrooms': bathrooms,
    'price': price.round(0)
})

df.head()


## 3. Explore the Data

In [ ]:
df.describe()


In [ ]:
# Check correlations with price
df.corr()['price'].sort_values(ascending=False)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].scatter(df['square_footage'], df['price'], alpha=0.5)
axes[0].set_xlabel('Square Footage')
axes[0].set_ylabel('Price')
axes[0].set_title('Square Footage vs Price')

axes[1].scatter(df['bedrooms'], df['price'], alpha=0.5)
axes[1].set_xlabel('Bedrooms')
axes[1].set_title('Bedrooms vs Price')

axes[2].scatter(df['bathrooms'], df['price'], alpha=0.5)
axes[2].set_xlabel('Bathrooms')
axes[2].set_title('Bathrooms vs Price')

plt.tight_layout()
plt.show()


## 4. Split into Train / Test Sets

In [ ]:
X = df[['square_footage', 'bedrooms', 'bathrooms']]
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training samples: {len(X_train)}')
print(f'Test samples: {len(X_test)}')


## 5. Train the Linear Regression Model

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

print('Model coefficients:')
for feature, coef in zip(X.columns, model.coef_):
    print(f'  {feature}: {coef:,.2f}')
print(f'Intercept: {model.intercept_:,.2f}')


**Interpreting the coefficients:** each coefficient tells you how much the predicted price changes for a one-unit increase in that feature, holding the others constant. For example, the square footage coefficient is roughly the extra price per additional square foot.

## 6. Evaluate the Model

In [ ]:
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f'Mean Absolute Error (MAE): {mae:,.2f}')
print(f'Root Mean Squared Error (RMSE): {rmse:,.2f}')
print(f'R² Score: {r2:.4f}')


In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual Price')
plt.ylabel('Predicted Price')
plt.title('Actual vs Predicted Prices')
plt.tight_layout()
plt.show()


## 7. Predict Price for a New House

Try changing the values below to see how the prediction changes.

In [ ]:
new_house = pd.DataFrame({
    'square_footage': [2200],
    'bedrooms': [3],
    'bathrooms': [2]
})

predicted_price = model.predict(new_house)[0]
print(f'Predicted price: ${predicted_price:,.2f}')


## Summary

- We trained a **Linear Regression** model to predict house prices from square footage, bedrooms, and bathrooms.
- The model's coefficients show the estimated dollar impact of each feature.
- We evaluated it using MAE, RMSE, and R² on a held-out test set.
- **Next steps to make this more robust for a real submission:**
  - Swap in a real dataset (e.g. Kaggle's House Prices dataset)
  - Check for and handle outliers/missing values
  - Try scaling features and comparing with `Ridge`/`Lasso` regression
  - Add more features if available (lot size, year built, location, etc.)
